# AC-OPF Predict-Then-Optimize — Base Case

## What this notebook does
This is the **simplest possible** predict-then-optimize (PTO) pipeline for AC-OPF:

```
Features x  →  [MLP]  →  predicted demand d̂  →  [AC-OPF]  →  dispatch z*(d̂)
                                                                      ↓
                                         evaluate cost vs. optimal at true demand d
```

The model is trained with plain **MSE loss** — no task-loss, no SPO+. The point is to
establish a clean baseline and show that even good demand predictions can introduce
meaningful dispatch sub-optimality, which motivates the task-loss work in the companion notebook.

## AC-OPF: Why the SDP relaxation?
AC-OPF is a **non-convex QCQP** (quadratically constrained quadratic program) — NP-hard
in the worst case.  In practice it solves in milliseconds for test cases, but the NLP formulation
has two problems for PTO research:

1. Local solvers (scipy SLSQP) may find different local optima for different demand inputs, making regret comparisons noisy.
2. Lagrange multipliers (LMPs) are not cleanly exposed by scipy's interface.

The **SDP relaxation** (Lavaei & Low 2012) fixes both:

$$\text{Relax } W = VV^H \;(\text{rank-1}) \quad\to\quad W \succeq 0 \;(\text{PSD})$$

This is a convex semidefinite program — global optimum guaranteed, exact duals accessible,
and **empirically tight** (rank-1 solution recovered) on networks like IEEE 14-bus.

**Solver stack:** CVXPY + CLARABEL (free, ships with CVXPY, handles complex Hermitian SDPs).

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import cvxpy as cp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import StandardScaler
import warnings, copy, time, os
from tqdm.notebook import trange, tqdm

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

BASEMVA = 100.0
DEVICE  = torch.device('cpu')

# Check CLARABEL is available
solvers = cp.installed_solvers()
print(f'CVXPY {cp.__version__}  |  Available solvers: {solvers}')
assert 'CLARABEL' in solvers or 'SCS' in solvers, \
    'Need CLARABEL or SCS — run: pip install clarabel'
OPF_SOLVER = cp.CLARABEL if 'CLARABEL' in solvers else cp.SCS
print(f'Using OPF solver: {OPF_SOLVER}')

(CVXPY) Apr 12 08:24:11 PM: Encountered unexpected exception importing solver GLOP:
OSError(22, 'The specified procedure could not be found', None, 127, None)
(CVXPY) Apr 12 08:24:11 PM: Encountered unexpected exception importing solver PDLP:
OSError(22, 'The specified procedure could not be found', None, 127, None)
(CVXPY) Apr 12 08:24:12 PM: Encountered unexpected exception importing solver GLOP:
OSError(22, 'The specified procedure could not be found', None, 127, None)
(CVXPY) Apr 12 08:24:12 PM: Encountered unexpected exception importing solver PDLP:
OSError(22, 'The specified procedure could not be found', None, 127, None)


CVXPY 1.7.2  |  Available solvers: ['CLARABEL', 'CVXOPT', 'ECOS', 'ECOS_BB', 'GLPK', 'GLPK_MI', 'GUROBI', 'OSQP', 'SCIPY', 'SCS']
Using OPF solver: CLARABEL


## 1. IEEE 14-Bus Network Data
Standard MATPOWER `case14` values.  All quantities in physical units (MW, MVAR) —
conversion to per-unit happens inside the solver.

In [2]:
# ── Bus: [bus_id, type(1=PQ,2=PV,3=slack), Pd_MW, Qd_MVAR, Vmax_pu, Vmin_pu] ──
BUS = np.array([
    [1,  3,  0.0,   0.0,  1.06, 0.94],
    [2,  2, 21.7,  12.7,  1.06, 0.94],
    [3,  2, 94.2,  19.0,  1.06, 0.94],
    [4,  1, 47.8,  -3.9,  1.06, 0.94],
    [5,  1,  7.6,   1.6,  1.06, 0.94],
    [6,  2, 11.2,   7.5,  1.06, 0.94],
    [7,  1,  0.0,   0.0,  1.06, 0.94],
    [8,  2,  0.0,   0.0,  1.06, 0.94],
    [9,  1, 29.5,  16.6,  1.06, 0.94],
    [10, 1,  9.0,   5.8,  1.06, 0.94],
    [11, 1,  3.5,   1.8,  1.06, 0.94],
    [12, 1,  6.1,   1.6,  1.06, 0.94],
    [13, 1, 13.5,   5.8,  1.06, 0.94],
    [14, 1, 14.9,   5.0,  1.06, 0.94],
])

# ── Generators: [bus_1idx, Pmax_MW, Pmin_MW, Qmax_MVAR, Qmin_MVAR] ──
GEN = np.array([
    [1, 332.4,  0.0,  10.0, -140.0],
    [2, 140.0,  0.0,  50.0,  -40.0],
    [3, 100.0,  0.0,  40.0,    0.0],
    [6, 100.0,  0.0,  24.0,   -6.0],
    [8, 100.0,  0.0,  24.0,   -6.0],
])

# ── Generator costs: [c2 $/MW²h, c1 $/MWh, c0 $/h] ──
GENCOST = np.array([
    [0.0430, 20.0, 0.0],
    [0.0250, 18.0, 0.0],
    [0.0100, 25.0, 0.0],
    [0.0150, 22.0, 0.0],
    [0.0200, 24.0, 0.0],
])

# ── Branches: [from, to, r_pu, x_pu, b_shunt_pu, Smax_MVA] ──
BRANCH = np.array([
    [ 1,  2, 0.01938, 0.05917, 0.0528, 9900],
    [ 1,  5, 0.05403, 0.22304, 0.0492, 9900],
    [ 2,  3, 0.04699, 0.19797, 0.0438, 9900],
    [ 2,  4, 0.05811, 0.17632, 0.0340, 9900],
    [ 2,  5, 0.05695, 0.17388, 0.0346, 9900],
    [ 3,  4, 0.06701, 0.17103, 0.0128, 9900],
    [ 4,  5, 0.01335, 0.04211, 0.0000, 9900],
    [ 4,  7, 0.00000, 0.20912, 0.0000, 9900],
    [ 4,  9, 0.00000, 0.55618, 0.0000, 9900],
    [ 5,  6, 0.00000, 0.25202, 0.0000, 9900],
    [ 6, 11, 0.09498, 0.19890, 0.0000, 9900],
    [ 6, 12, 0.12291, 0.25581, 0.0000, 9900],
    [ 6, 13, 0.06615, 0.13027, 0.0000, 9900],
    [ 7,  8, 0.00000, 0.17615, 0.0000, 9900],
    [ 7,  9, 0.00000, 0.11001, 0.0000, 9900],
    [ 9, 10, 0.03181, 0.08450, 0.0000, 9900],
    [ 9, 14, 0.12711, 0.27038, 0.0000, 9900],
    [10, 11, 0.08205, 0.19207, 0.0000, 9900],
    [12, 13, 0.22092, 0.19988, 0.0000, 9900],
    [13, 14, 0.17093, 0.34802, 0.0000, 9900],
])
# NOTE: Line limits are all 9900 (unlimited) in the base case — the SDP relaxation
# without line limits focuses on generation cost optimality and voltage feasibility.
# Adding thermal limits is a natural next step.

N_BUS    = len(BUS)
N_GEN    = len(GEN)
N_BRANCH = len(BRANCH)
GEN_BUS  = (GEN[:, 0] - 1).astype(int)          # 0-indexed generator bus
LOAD_MASK  = BUS[:, 2] > 0.0                      # buses with non-zero base load
LOAD_BUSES = np.where(LOAD_MASK)[0]               # 0-indexed
N_LOAD   = len(LOAD_BUSES)

# Bus-level Q/P ratio from base case (preserves reactive power structure)
BASE_P = BUS[:, 2]   # MW
BASE_Q = BUS[:, 3]   # MVAR
QP_RATIO = np.where(BASE_P > 0, BASE_Q / BASE_P, 0.0)  # Q per MW at each bus

print(f'Buses: {N_BUS}  |  Generators: {N_GEN}  |  Load buses: {N_LOAD}')
print(f'Base total load: P={BASE_P.sum():.1f} MW,  Q={BASE_Q.sum():.1f} MVAR')

Buses: 14  |  Generators: 5  |  Load buses: 11
Base total load: P=259.0 MW,  Q=73.5 MVAR


## 2. Complex Admittance Matrix (Ybus)
The AC power flow equations are written in terms of the **Ybus** matrix:
$$S_i = V_i \sum_j Y_{ij}^* V_j^* \quad\Rightarrow\quad P_i + jQ_i = \sum_j Y_{ij}^* W_{ij}$$
where $W_{ij} = V_i V_j^*$ are elements of the voltage outer-product matrix $W = VV^H$.

In [ ]:
def build_Ybus():
    """
    Build the complex bus admittance matrix from branch data.
    Includes series admittance and shunt charging susceptance (b/2 at each end).
    Transformer tap ratios for branches 8, 9, 10 (IEEE 14-bus).
    """
    TAP = np.ones(N_BRANCH)          # off-nominal transformer tap ratios
    TAP[7]  = 0.978                  # branch 4→7
    TAP[8]  = 0.969                  # branch 4→9
    TAP[9]  = 0.932                  # branch 5→6

    Y = np.zeros((N_BUS, N_BUS), dtype=complex)

    for k, br in enumerate(BRANCH):
        i, j   = int(br[0]) - 1, int(br[1]) - 1
        r, x   = br[2], br[3]
        b_sh   = br[4]               # shunt susceptance (total line charging)
        tau    = TAP[k]              # tap ratio
        y_s    = 1.0 / complex(r, x) # series admittancea

        # Standard π-model with off-nominal tap (Kundur / Bergen & Vittal convention)
        Y[i, i] += y_s / tau**2 + 1j * b_sh / 2
        Y[j, j] += y_s            + 1j * b_sh / 2
        Y[i, j] -= y_s / tau
        Y[j, i] -= np.conj(y_s / tau)   # conj for non-unity tap (asymmetric)

    return Y

YBUS = build_Ybus()

print(f'Ybus shape: {YBUS.shape}  dtype: {YBUS.dtype}')
print(f'Diagonal (self-admittance magnitudes): '
      f'{np.abs(np.diag(YBUS)).round(2)}')

Ybus shape: (14, 14)  dtype: complex128
Diagonal (self-admittance magnitudes): [20.36 31.73 10.31 40.06 36.8  18.55 19.55  5.68 24.86 15.86  9.32  6.75
 12.61  5.93]


## 3. SDP-Relaxed AC-OPF Solver

### Problem formulation
**Variables:** $W \in \mathbb{C}^{n \times n}$ (Hermitian PSD), $P_g \in \mathbb{R}^{n_g}$, $Q_g \in \mathbb{R}^{n_g}$

$$\min_{W, P_g, Q_g} \sum_j \left( c_{2j} P_{gj}^2 + c_{1j} P_{gj} + c_{0j} \right)$$

subject to

$$\sum_j Y_{ij}^* W_{ij} = P_{gi} - P_{di} + j(Q_{gi} - Q_{di}) \quad \forall i \quad \text{(power balance)}$$
$$V_{i,\min}^2 \leq W_{ii} \leq V_{i,\max}^2 \quad \forall i \quad \text{(voltage limits)}$$
$$P_{g,\min} \leq P_g \leq P_{g,\max}, \quad Q_{g,\min} \leq Q_g \leq Q_{g,\max}$$
$$W \succeq 0 \quad \text{(SDP relaxation of } W = VV^H \text{)}$$

### Tightness check
If the relaxation is **tight**, then $\text{rank}(W^*) = 1$, meaning $W^* = V^* (V^*)^H$ for some
feasible voltage vector $V^*$.  We recover voltages as the leading eigenvector of $W^*$.
For IEEE 14-bus without line limits, tightness holds in virtually all demand scenarios.

In [4]:
# Pre-compute generator-to-bus lookup once
_GEN_AT_BUS = {b: [] for b in range(N_BUS)}
for j, bus in enumerate(GEN_BUS):
    _GEN_AT_BUS[bus].append(j)


def solve_acopf(p_demand_mw, q_demand_mvar, verbose=False):
    """
    Solve the SDP-relaxed AC-OPF (Lavaei & Low 2012).

    Args:
        p_demand_mw   : ndarray (N_BUS,)  active demand at every bus  [MW]
        q_demand_mvar : ndarray (N_BUS,)  reactive demand at every bus [MVAR]

    Returns dict:
        cost      $/h    optimal generation cost
        pg_mw     MW     active generation dispatch (n_gen,)
        qg_mvar   MVAR   reactive generation dispatch (n_gen,)
        vm        pu     voltage magnitudes recovered from W (n_bus,)
        lmps_p    $/MWh  active LMPs at each bus (dual of P balance)
        lmps_q    $/MVARh reactive LMPs at each bus (dual of Q balance)
        rank1_gap float  ratio of 2nd/1st eigenvalue of W (0 = perfectly tight)
        success   bool
    """
    pd_pu = p_demand_mw  / BASEMVA
    qd_pu = q_demand_mvar / BASEMVA

    # ── Decision variables ──
    W  = cp.Variable((N_BUS, N_BUS), hermitian=True)   # outer-product relaxation
    pg = cp.Variable(N_GEN)    # per-unit active generation
    qg = cp.Variable(N_GEN)    # per-unit reactive generation

    # ── Cost ($/h, quadratic in Pg_MW = pg * BASEMVA) ──
    cost_expr = cp.sum([
        GENCOST[j, 0] * (pg[j] * BASEMVA)**2
        + GENCOST[j, 1] * (pg[j] * BASEMVA)
        + GENCOST[j, 2]
        for j in range(N_GEN)
    ])

    # ── Power balance constraints (one per bus, P and Q) ──
    p_balance_cons = []
    q_balance_cons = []

    constraints = [W >> 0]   # SDP: W positive semidefinite

    for i in range(N_BUS):
        # S_i = Σ_j  Y_ij^*  ·  W_ij
        S_i = cp.sum([np.conj(YBUS[i, j]) * W[i, j] for j in range(N_BUS)])

        pg_i = cp.sum([pg[k] for k in _GEN_AT_BUS[i]]) if _GEN_AT_BUS[i] else 0.0
        qg_i = cp.sum([qg[k] for k in _GEN_AT_BUS[i]]) if _GEN_AT_BUS[i] else 0.0

        pb = (cp.real(S_i) == pg_i - pd_pu[i])
        qb = (cp.imag(S_i) == qg_i - qd_pu[i])
        p_balance_cons.append(pb)
        q_balance_cons.append(qb)
        constraints += [pb, qb]

    # ── Voltage magnitude limits  (W_ii = |V_i|²) ──
    for i in range(N_BUS):
        constraints += [
            cp.real(W[i, i]) >= BUS[i, 5]**2,   # Vmin²
            cp.real(W[i, i]) <= BUS[i, 4]**2,   # Vmax²
        ]

    # ── Generator capacity limits ──
    for j in range(N_GEN):
        constraints += [
            pg[j] >= GEN[j, 2] / BASEMVA,   # Pmin
            pg[j] <= GEN[j, 1] / BASEMVA,   # Pmax
            qg[j] >= GEN[j, 4] / BASEMVA,   # Qmin
            qg[j] <= GEN[j, 3] / BASEMVA,   # Qmax
        ]

    # ── Solve ──
    prob = cp.Problem(cp.Minimize(cost_expr), constraints)
    prob.solve(solver=OPF_SOLVER, verbose=verbose)

    if prob.status not in ('optimal', 'optimal_inaccurate'):
        return {'success': False, 'status': prob.status}

    # ── Extract solution ──
    W_val = W.value                                    # complex n×n matrix
    eigvals = np.linalg.eigvalsh(W_val)[::-1]         # sorted descending (real, Hermitian)
    rank1_gap = eigvals[1] / (eigvals[0] + 1e-12)     # ≈ 0 means rank-1 (tight)

    # Recover voltages from leading eigenvector
    _, evecs = np.linalg.eigh(W_val)
    v_lead = evecs[:, -1] * np.sqrt(eigvals[0])       # complex voltage vector
    vm = np.abs(v_lead)                                # voltage magnitudes (pu)

    # LMPs: dual of equality constraint is ∂cost/∂(RHS)
    # Balance: Re(Σ Y_ij* W_ij) = pg_i - pd_i  → dual / BASEMVA gives $/MWh
    lmps_p = np.array([c.dual_value for c in p_balance_cons], dtype=float) / BASEMVA
    lmps_q = np.array([c.dual_value for c in q_balance_cons], dtype=float) / BASEMVA

    return {
        'cost':      float(prob.value),
        'pg_mw':     pg.value * BASEMVA,
        'qg_mvar':   qg.value * BASEMVA,
        'vm':        vm,
        'lmps_p':    lmps_p,           # $/MWh at each bus
        'lmps_q':    lmps_q,           # $/MVARh at each bus
        'rank1_gap': float(rank1_gap), # closeness to AC-feasibility (want ≈ 0)
        'W':         W_val,
        'success':   True,
    }

In [5]:
# ── Verify solver on the base-case (known) load ──
p0 = BUS[:, 2].copy()   # base-case P demand (MW)
q0 = BUS[:, 3].copy()   # base-case Q demand (MVAR)

print('Solving base-case AC-OPF (SDP relaxation)...')
t0  = time.time()
r0  = solve_acopf(p0, q0)
dt  = time.time() - t0

assert r0['success'], f'Solver failed on base case: {r0.get("status")}'
print(f'\nSolved in {dt:.2f}s')
print(f'Optimal cost : {r0["cost"]:>10,.2f} $/h')
print(f'Generator dispatch (MW):')
for j in range(N_GEN):
    print(f'  Gen {j+1} (bus {int(GEN[j,0])}): Pg = {r0["pg_mw"][j]:.2f} MW, '
          f'Qg = {r0["qg_mvar"][j]:.2f} MVAR')
print(f'Voltage magnitudes (pu): {r0["vm"].round(4)}')
print(f'Rank-1 gap: {r0["rank1_gap"]:.2e}  (< 1e-3 → relaxation is tight)')
print(f'LMPs at load buses ($/MWh): {r0["lmps_p"][LOAD_BUSES].round(3)}')

# Sanity: total generation ≈ total load (losses from line charging)
print(f'\nTotal Pg: {r0["pg_mw"].sum():.2f} MW  |  Total load: {p0.sum():.2f} MW')

Solving base-case AC-OPF (SDP relaxation)...


AssertionError: Solver failed on base case: infeasible

## 4. Synthetic Demand Dataset

Each sample is one hour.  The feature vector captures everything an operator would observe
*before* knowing the actual demand:

| Feature | Description |
|---------|-------------|
| `sin/cos(hour/24)` | Diurnal cycle |
| `sin/cos(day/7)` | Weekly cycle |
| `sin/cos(day_of_year/365)` | Seasonal cycle |
| Temperature | Synthetic seasonal + diurnal (normalised) |
| AI-DC state | Markov-chain: 1=training, 0=idle (bus 9) |
| Weekend flag | Binary |

**Reactive demand** is scaled from the active demand using the bus-specific base-case Q/P
ratio — this preserves the real network's reactive power structure without adding a
second prediction head to the model.

In [ ]:
def generate_dataset(n_samples=1500, seed=42):
    """
    Returns:
        X       (n, 10)      feature matrix
        P_true  (n, N_BUS)   active demand at ALL buses [MW]  (0 at non-load buses)
        Q_true  (n, N_BUS)   reactive demand at ALL buses [MVAR]
    """
    rng = np.random.default_rng(seed)
    base_p = BASE_P[LOAD_BUSES]   # shape (N_LOAD,)

    # ── AI datacenter Markov chain (bus 9, LOAD_BUSES index 4) ──
    DC_IDX   = list(LOAD_BUSES).index(8)   # bus 9 is index 8 (0-based)
    dc_state = np.zeros(n_samples, dtype=int)
    s = 1
    for t in range(n_samples):
        dc_state[t] = s
        s = 0 if (s == 1 and rng.random() < 0.02) else (1 if (s == 0 and rng.random() < 0.70) else s)

    hours       = np.arange(n_samples)
    hour_of_day = hours % 24
    day_of_week = (hours // 24) % 7
    day_of_year = (hours // 24) % 365
    temperature = (15 + 12*np.sin(2*np.pi*(hours-2190)/8760)
                      + 5*np.sin(2*np.pi*hours/24)
                      + rng.normal(0, 2, n_samples))

    X = np.column_stack([
        np.sin(2*np.pi*hour_of_day/24),
        np.cos(2*np.pi*hour_of_day/24),
        np.sin(2*np.pi*day_of_week/7),
        np.cos(2*np.pi*day_of_week/7),
        np.sin(2*np.pi*day_of_year/365),
        np.cos(2*np.pi*day_of_year/365),
        temperature / 30.0,
        dc_state.astype(float),
        (day_of_week >= 5).astype(float),
        (temperature > 28).astype(float),
    ]).astype(np.float32)

    # ── Demand generation ──
    diurnal = 0.75 + 0.25*np.sin(2*np.pi*(hour_of_day-8)/24)
    weekly  = np.where(day_of_week >= 5, 0.85, 1.0)
    temp_f  = 1 + 0.008*np.maximum(temperature-18, 0) + 0.005*np.maximum(12-temperature, 0)
    mult    = (diurnal * weekly * temp_f)[:, None]   # (n, 1)

    # Spatially correlated noise
    A    = rng.standard_normal((N_LOAD, N_LOAD))
    corr = (A @ A.T) / N_LOAD + 0.5*np.eye(N_LOAD)
    L    = np.linalg.cholesky(corr)
    eps  = (L @ rng.standard_normal((N_LOAD, n_samples))).T * 0.06  # (n, N_LOAD)

    P_load = base_p[None, :] * mult * (1 + eps)

    # AI datacenter extra load on DC bus
    P_load[:, DC_IDX] += np.where(dc_state == 1, 20.0, 4.0)
    P_load = np.maximum(P_load, 0.5)

    # Map to full bus vectors
    P_true = np.zeros((n_samples, N_BUS), dtype=np.float32)
    Q_true = np.zeros((n_samples, N_BUS), dtype=np.float32)
    P_true[:, LOAD_BUSES] = P_load
    Q_true[:, LOAD_BUSES] = P_load * QP_RATIO[LOAD_BUSES][None, :]  # scale Q with P

    return X, P_true, Q_true


X_all, P_all, Q_all = generate_dataset(n_samples=1500)
print(f'Dataset: X={X_all.shape}, P={P_all.shape}, Q={Q_all.shape}')
print(f'P range: {P_all[P_all>0].min():.1f} – {P_all.max():.1f} MW')
print(f'Total load: {P_all.sum(1).mean():.1f} ± {P_all.sum(1).std():.1f} MW (mean ± std)')

In [ ]:
CACHE = 'base_cache.npz'

def presolve_all(X, P, Q, cache=CACHE):
    """Pre-solve AC-OPF for every scenario and cache results."""
    if os.path.exists(cache):
        print(f'Loading cache from {cache}...')
        d = np.load(cache)
        return d['costs'], d['lmps_p'], d['lmps_q'], d['vm'], d['ok']

    n = len(P)
    costs  = np.full(n, np.nan)
    lmps_p = np.zeros((n, N_BUS), dtype=np.float32)
    lmps_q = np.zeros((n, N_BUS), dtype=np.float32)
    vm_all = np.zeros((n, N_BUS), dtype=np.float32)
    ok     = np.zeros(n, dtype=bool)

    print(f'Pre-solving AC-OPF (SDP) for {n} scenarios...')
    t0 = time.time()
    for i in trange(n):
        r = solve_acopf(P[i], Q[i])
        if r['success']:
            costs[i]   = r['cost']
            lmps_p[i]  = r['lmps_p']
            lmps_q[i]  = r['lmps_q']
            vm_all[i]  = r['vm']
            ok[i]      = True

    print(f'Done in {time.time()-t0:.0f}s  |  '
          f'Success {ok.mean()*100:.1f}%  |  '
          f'Mean cost {np.nanmean(costs):.1f} $/h')
    np.savez(cache, costs=costs, lmps_p=lmps_p, lmps_q=lmps_q, vm=vm_all, ok=ok)
    return costs, lmps_p, lmps_q, vm_all, ok


COSTS, LMPS_P, LMPS_Q, VM_ALL, SOLVE_OK = presolve_all(X_all, P_all, Q_all)

# Keep only feasible scenarios
idx_ok  = np.where(SOLVE_OK)[0]
X_ok    = X_all[idx_ok]
P_ok    = P_all[idx_ok]
Q_ok    = Q_all[idx_ok]
C_ok    = COSTS[idx_ok]
LP_ok   = LMPS_P[idx_ok]

print(f'\nFeasible scenarios: {len(idx_ok)}/{len(idx_ok)}')
print(f'Cost range: {C_ok.min():.0f} – {C_ok.max():.0f} $/h')

## 5. Train/Val/Test Split

In [ ]:
rng_s = np.random.default_rng(99)
perm  = rng_s.permutation(len(idx_ok))

n_train = int(0.70 * len(idx_ok))
n_val   = int(0.15 * len(idx_ok))

train_idx = perm[:n_train]
val_idx   = perm[n_train:n_train+n_val]
test_idx  = perm[n_train+n_val:]

print(f'Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}')

# Feature scaling
scaler  = StandardScaler()
X_sc    = scaler.fit_transform(X_ok).astype(np.float32)

# Target: active power at load buses only (MW, raw scale)
P_target = P_ok[:, LOAD_BUSES].astype(np.float32)   # (N, N_LOAD)
P_NORM   = P_target.mean(0) + 1e-6                   # per-bus normaliser for loss

def to_tensor(idx):
    x = torch.tensor(X_sc[idx]).to(DEVICE)
    p = torch.tensor(P_target[idx]).to(DEVICE)
    return x, p

Xt, Pt = to_tensor(train_idx)
Xv, Pv = to_tensor(val_idx)
Xe, Pe = to_tensor(test_idx)

## 6. MLP Demand Prediction Model
Input: 10 contextual features.  Output: predicted active demand at each of the
$N_{\text{load}}$ load buses (MW).  `Softplus` output ensures predictions are
strictly positive (physical demand).  Trained with normalised MSE.

In [ ]:
class DemandMLP(nn.Module):
    """Feature → active demand [MW] at each load bus."""
    def __init__(self, n_in=10, n_out=N_LOAD, hidden=(128, 128, 64)):
        super().__init__()
        layers, d = [], n_in
        for h in hidden:
            layers += [nn.Linear(d, h), nn.LayerNorm(h), nn.ReLU()]
            d = h
        layers += [nn.Linear(d, n_out), nn.Softplus(beta=5)]  # strictly positive
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)   # (batch, N_LOAD) in MW


def norm_mse(pred, true):
    """Normalised MSE — scale-invariant across buses."""
    norm = torch.tensor(P_NORM, device=DEVICE)
    return ((pred - true) / norm).pow(2).mean()


def train(model, epochs=300, lr=3e-4, batch=128, patience=30):
    opt    = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loader = DataLoader(TensorDataset(Xt, Pt), batch_size=batch, shuffle=True)
    hist   = {'train': [], 'val': []}
    best_v, best_s, wait = np.inf, None, 0

    for ep in trange(epochs, desc='Train'):
        model.train()
        tl = 0.0
        for xb, pb in loader:
            opt.zero_grad()
            loss = norm_mse(model(xb), pb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tl += loss.item()
        sched.step()
        model.eval()
        with torch.no_grad():
            vl = norm_mse(model(Xv), Pv).item()
        hist['train'].append(tl / len(loader))
        hist['val'].append(vl)
        if vl < best_v:
            best_v, best_s, wait = vl, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_s)
    return hist


model = DemandMLP()
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
hist = train(model)
print(f'Best val normalised MSE: {min(hist["val"]):.5f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hist['train'], label='Train', color='steelblue')
ax.plot(hist['val'],   label='Val',   color='tomato')
ax.set_yscale('log')
ax.set_xlabel('Epoch')
ax.set_ylabel('Normalised MSE (log scale)')
ax.set_title('MLP Training Curve (MSE loss)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('training_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Evaluation: Predict → AC-OPF → Measure Regret

For each test scenario we run the full pipeline:

1. **Predict** $\hat{P}$ from features $x$
2. **Derive** $\hat{Q} = \hat{P} \times \text{(Q/P ratio)}$
3. **Solve** AC-OPF$(\hat{P}, \hat{Q})$ → dispatch cost $v(\hat{d})$
4. **Compare** to optimal cost at true demand $v(d)$ → **regret** $= v(\hat{d}) - v(d)$

Key metrics:
- **Mean regret** ($/h): average extra cost from using predicted vs. true demand
- **Regret as % of optimal cost**: scale-normalised sub-optimality
- **Infeasibility rate**: % of test scenarios where AC-OPF fails on predicted demand
- **MAE** (MW): raw prediction accuracy for reference

In [ ]:
model.eval()
with torch.no_grad():
    pred_mw = model(Xe).cpu().numpy()   # (N_test, N_LOAD) — predicted active demand

true_mw    = P_ok[test_idx][:, LOAD_BUSES]   # (N_test, N_LOAD)
cost_true  = C_ok[test_idx]                  # optimal cost at TRUE demand

records = []
print('Solving AC-OPF with predicted demands on test set...')
for i in trange(len(test_idx)):
    p_pred_bus = np.zeros(N_BUS)
    p_pred_bus[LOAD_BUSES] = np.maximum(pred_mw[i], 0.5)           # clip to positive
    q_pred_bus = p_pred_bus * QP_RATIO                              # scale Q from predicted P

    r = solve_acopf(p_pred_bus, q_pred_bus)

    if r['success']:
        regret = max(0.0, r['cost'] - cost_true[i])                # non-negative by definition
        pct    = regret / cost_true[i] * 100
        records.append({
            'cost_pred':   r['cost'],
            'cost_true':   cost_true[i],
            'regret':      regret,
            'regret_pct':  pct,
            'rank1_gap':   r['rank1_gap'],
            'mae_p':       np.abs(pred_mw[i] - true_mw[i]).mean(),
            'feasible':    True,
        })
    else:
        records.append({
            'cost_pred':  np.nan, 'cost_true': cost_true[i],
            'regret':     np.nan, 'regret_pct': np.nan,
            'rank1_gap':  np.nan, 'mae_p': np.abs(pred_mw[i] - true_mw[i]).mean(),
            'feasible':   False,
        })

df = pd.DataFrame(records)
df_feas = df[df['feasible']]

print('\n=== Test Set Results ===')
print(f"Feasibility rate        : {df['feasible'].mean()*100:.1f}%")
print(f"Mean regret             : {df_feas['regret'].mean():.3f} $/h")
print(f"Regret as % of opt cost : {df_feas['regret_pct'].mean():.3f}%")
print(f"95th-pct regret         : {df_feas['regret'].quantile(0.95):.3f} $/h")
print(f"Max regret              : {df_feas['regret'].max():.3f} $/h")
print(f"Mean demand MAE         : {df_feas['mae_p'].mean():.2f} MW")
print(f"Mean rank-1 gap         : {df_feas['rank1_gap'].mean():.2e}  (< 1e-3 → tight)")

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)

# ── 1. Predicted vs. True demand (scatter) ──
ax0 = fig.add_subplot(gs[0, 0])
for b_idx in range(N_LOAD):
    ax0.scatter(true_mw[:, b_idx], pred_mw[:, b_idx], s=4, alpha=0.3)
lim = max(true_mw.max(), pred_mw.max()) * 1.05
ax0.plot([0, lim], [0, lim], 'k--', linewidth=1)
ax0.set_xlabel('True demand (MW)')
ax0.set_ylabel('Predicted demand (MW)')
ax0.set_title('Predicted vs. True Demand\n(all load buses, test set)', fontsize=11)
ax0.grid(alpha=0.3)

# ── 2. OPF cost: predicted demand vs. true demand ──
ax1 = fig.add_subplot(gs[0, 1])
ax1.scatter(df_feas['cost_true'], df_feas['cost_pred'], s=8, alpha=0.5, color='steelblue')
lim2 = max(df_feas['cost_true'].max(), df_feas['cost_pred'].max()) * 1.02
ax1.plot([0, lim2], [0, lim2], 'k--', linewidth=1, label='Zero regret')
ax1.set_xlabel('OPF cost at TRUE demand ($/h)')
ax1.set_ylabel('OPF cost at PREDICTED demand ($/h)')
ax1.set_title('AC-OPF Cost: Predicted vs. Optimal\n(points above diagonal = positive regret)', fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# ── 3. Regret distribution ──
ax2 = fig.add_subplot(gs[0, 2])
ax2.hist(df_feas['regret'], bins=30, color='tomato', edgecolor='white', alpha=0.85)
ax2.axvline(df_feas['regret'].mean(), color='black', linestyle='--',
            label=f'Mean {df_feas["regret"].mean():.2f} $/h')
ax2.axvline(df_feas['regret'].quantile(0.95), color='gray', linestyle=':',
            label=f'P95 {df_feas["regret"].quantile(0.95):.2f} $/h')
ax2.set_xlabel('Regret ($/h)')
ax2.set_ylabel('Count')
ax2.set_title('OPF Regret Distribution (test set)', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

# ── 4. Regret vs. demand prediction error ──
ax3 = fig.add_subplot(gs[1, 0])
ax3.scatter(df_feas['mae_p'], df_feas['regret'], s=8, alpha=0.5, color='purple')
ax3.set_xlabel('Demand prediction MAE (MW)')
ax3.set_ylabel('OPF regret ($/h)')
ax3.set_title('Prediction Error → Regret\n(low correlation = MSE ≠ task loss)', fontsize=11)
corr = np.corrcoef(df_feas['mae_p'], df_feas['regret'])[0,1]
ax3.text(0.05, 0.92, f'Pearson r = {corr:.3f}', transform=ax3.transAxes, fontsize=10)
ax3.grid(alpha=0.3)

# ── 5. LMP heatmap (true demand, test scenarios) ──
ax4 = fig.add_subplot(gs[1, 1])
lmp_show = LP_ok[test_idx[:60]][:, LOAD_BUSES]
im = ax4.imshow(lmp_show.T, aspect='auto', cmap='RdYlGn_r')
ax4.set_xlabel('Test scenario')
ax4.set_ylabel('Load bus')
ax4.set_yticks(range(N_LOAD))
ax4.set_yticklabels([f'B{int(BUS[b,0])}' for b in LOAD_BUSES], fontsize=8)
ax4.set_title('AC LMPs — 60 test scenarios\n(vary = interesting for PTO)', fontsize=11)
plt.colorbar(im, ax=ax4, label='LMP ($/MWh)')

# ── 6. Rank-1 gap (SDP tightness) ──
ax5 = fig.add_subplot(gs[1, 2])
ax5.hist(np.log10(df_feas['rank1_gap'] + 1e-12), bins=30,
         color='steelblue', edgecolor='white', alpha=0.85)
ax5.axvline(-3, color='red', linestyle='--', label='Tightness threshold (1e-3)')
ax5.set_xlabel('log₁₀(rank-1 gap)')
ax5.set_ylabel('Count')
ax5.set_title('SDP Relaxation Tightness\n(left of red = AC-feasible solution recovered)', fontsize=11)
ax5.legend(fontsize=9)
ax5.grid(alpha=0.3)

plt.suptitle('AC-OPF PTO Base Case — MSE-Trained Model Evaluation', fontsize=13, y=1.01)
plt.savefig('base_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Summary and What This Baseline Tells Us

### Reading the results

**Scatter plot (panel 4)** — Pearson correlation between demand prediction MAE and OPF regret.  
If this correlation is **low** (< 0.5), it means that *where* you make errors matters more than
*how big* they are — a prediction that's 5 MW off on a high-LMP bus costs more than
one that's 20 MW off on a low-LMP bus.  This is the core motivation for task-loss training.

**SDP tightness (panel 6)** — rank-1 gap near 0 confirms the SDP relaxation recovers a
true AC-feasible solution.  If rank-1 gap is large on some scenarios, it means the SDP
optimum is not achievable by any actual voltage vector — the relaxation is not tight there.

**LMP heatmap (panel 5)** — spatial and temporal variation in LMPs is what makes different
buses have different importance for prediction.  A scenario with uniform LMPs benefits
little from task-loss training; a scenario with high LMP variance benefits a lot.

### Next steps (companion notebook: `acopf_pto.ipynb`)
| Improvement | Mechanism |
|-------------|----------|
| **SPO+ loss** | Replace MSE with LMP(d_true) · d_hat — pre-computed, cheap |
| **Task loss** | Use LMP(d_hat) as gradient (requires re-solving SOCP each iteration) |
| **Dataset selection** | Oversample high-LMP scenarios (LMP-diversity strategy) |
| **Thermal limits** | Add line flow constraints to the SDP — makes congestion binding |
| **Q prediction** | Separate Q prediction head rather than fixed Q/P ratio |